# Step 6 — Full iterative holographic tomographic reconstruction

Single-GPU notebook version of `step6.py`. Reads parameters from `config_step6.conf`.

Runs the Bilinear-Hessian solver to iteratively refine the 3-D complex refractive-index
distribution (object), the illumination probe, and the sample positions.
Resumes from the latest checkpoint if one exists.

In [ ]:
import numpy as np
import cupy as cp
import h5py
from mpi4py import MPI
from holotomocupy.rec_mpi import Rec
from holotomocupy.config import parse_args
from holotomocupy.mpi_functions import MPIClass
from holotomocupy.reader import Reader, find_latest_checkpoint
from holotomocupy.writer import Writer
from holotomocupy.logger_config import logger, set_log_level
from holotomocupy.utils import *

cp.cuda.set_pinned_memory_allocator(None)

In [ ]:
config_file = 'config_step6.conf'
args = parse_args(config_file)

comm = MPI.COMM_SELF  # single-GPU notebook
args.comm = comm
set_log_level(args.log_level)

print(f'in_file  = {args.in_file}')
print(f'path_out = {args.path_out}')
print(f'n={args.n}  nobj={args.nobj}  ntheta={args.ntheta}  ndist={args.ndist}  bin={args.bin}')
print(f'niter={args.niter}  nchunk={args.nchunk}  paganin={args.paganin}')

## Setup MPI distribution, Reader, and Writer

In [ ]:
cl_mpi = MPIClass(comm, args.nzobj, args.ntheta, args.nobj, args.obj_dtype)

reader = Reader(
    args.in_file, comm,
    cl_mpi.st_obj, cl_mpi.end_obj, args.nzobj, args.nobj,
    cl_mpi.st_theta, cl_mpi.end_theta, args.ntheta,
    args.ndist, args.nz, args.n, args.obj_dtype,
    args.paganin, args.rotation_center_shift, args.start_theta, args.bin,
)
writer = Writer(
    args.path_out, comm,
    cl_mpi.st_obj, cl_mpi.end_obj, args.nzobj, args.nobj,
    cl_mpi.st_theta, cl_mpi.end_theta, args.ntheta,
    args.ndist, args.nz, args.n, args.obj_dtype,
)

args.energy                  = args.energy
args.focustodetectordistance = reader.focustodetectordistance
args.z1                      = reader.z1
args.detector_pixelsize      = reader.detector_pixelsize
args.theta                   = reader.theta

print(f'energy={args.energy} keV  z1={args.z1}  detector_pixelsize={args.detector_pixelsize}')

## Initialise reconstruction class

In [ ]:
cl = Rec(args)
cl.method = args.method
print(f'obj-range   [{cl.st_obj}:{cl.end_obj}), local {cl.end_obj-cl.st_obj} x {cl.nobj} x {cl.nobj}')
print(f'theta-range [{cl.st_theta}:{cl.end_theta}), local {cl.end_theta-cl.st_theta} x {cl.nzobj} x {cl.nobj}')

## Load measurements and flat-field reference

In [ ]:
reader.read_data(out=cl.data)
reader.read_ref(out=cl.ref)
print(f'data shape: {cl.data.shape}  ref shape: {cl.ref.shape}')

## Load initial variables (object, probe, positions)

Resumes from the latest checkpoint if one exists; otherwise uses the Paganin reconstruction from Step 5.

In [ ]:
ckpt = find_latest_checkpoint(args.path_out, args.start_iter)
if ckpt:
    print(f'Resuming from checkpoint: {ckpt}')
    reader.read_checkpoint(ckpt, out_obj=cl.vars['obj'], out_pos=cl.vars['pos'], out_prb=cl.vars['prb'])
else:
    print('Starting from Paganin initial guess')
    reader.read_obj(out=cl.vars['obj'])
    reader.read_pos(out=cl.vars['pos'])
    if args.prb_file:
        print(f'Loading probe from: {args.prb_file}  (dist index {args.ref_dist})')
        with h5py.File(args.prb_file, 'r') as _f:
            _amp   = _f['prb_amp'][args.ref_dist]
            _phase = _f['prb_phase'][args.ref_dist]
        cl.vars['prb'][:] = cp.array((_amp * np.exp(1j * _phase)).astype('complex64'))
    else:
        cl.vars['prb'][:] = 1
if args.pos_checkpoint:
    print(f'Overriding positions from: {args.pos_checkpoint}')
    reader.read_pos_checkpoint(args.pos_checkpoint, out=cl.vars['pos'])

## Visualise initial object and probe

In [ ]:
show = True
if show:
    nz2 = cl.vars['obj'].shape[0] // 2
    mshow(cl.vars['obj'][nz2].real, show)
    mshow_polar(cl.vars['prb'][0], show)

## Run iterative reconstruction

In [ ]:
vars = cl.BH(writer)

## Visualise final result

In [ ]:
show = True
if show:
    import matplotlib.pyplot as plt

    obj = cl.vars['obj']
    prb = cl.vars['prb']
    nz2 = obj.shape[0] // 2
    ny2 = obj.shape[1] // 2

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    slices = [obj[nz2], obj[:, ny2], obj[:, :, ny2]]
    names  = ['z-slice', 'y-slice', 'x-slice']
    for col, (sl, nm) in enumerate(zip(slices, names)):
        sl_np = sl.real.get() if hasattr(sl, 'get') else sl.real
        axes[0, col].imshow(sl_np, cmap='gray')
        axes[0, col].set_title(f'obj.real  {nm}')
        axes[0, col].axis('off')
        sl_np = sl.imag.get() if hasattr(sl, 'get') else sl.imag
        axes[1, col].imshow(sl_np, cmap='gray')
        axes[1, col].set_title(f'obj.imag  {nm}')
        axes[1, col].axis('off')
    plt.tight_layout()
    plt.show()

    mshow_polar(prb[0], show)